# Crop Disease Detection - EfficientNetB3 Training
Run this notebook in Google Colab with GPU enabled.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
import os, json, shutil, random, zipfile
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB3
from google.colab import files

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Upload your Dataset.zip
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('.')
print('Extracted:', zip_path)

In [ ]:
# Configuration
DATASET_DIR = 'Dataset'
IMG_SIZE = (300, 300)
BATCH_SIZE = 32
VAL_SPLIT = 0.2
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 15
LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-5
random.seed(42)

In [ ]:
# Split into train/val
base_dir = 'data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

classes = sorted(os.listdir(DATASET_DIR))
for cls in classes:
    src = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(src):
        continue
    images = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(images)
    split = int(len(images) * (1 - VAL_SPLIT))
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    for img in images[:split]:
        shutil.copy2(os.path.join(src, img), os.path.join(train_dir, cls, img))
    for img in images[split:]:
        shutil.copy2(os.path.join(src, img), os.path.join(val_dir, cls, img))
    print(f'{cls}: {split} train, {len(images)-split} val')

print(f'\nTotal classes: {len(classes)}')

In [ ]:
# Data generators
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
)

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
)

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)
val_gen = val_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

In [ ]:
# Build model
base_model = EfficientNetB3(include_top=False, weights='imagenet',
                            input_shape=(300, 300, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(300, 300, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(classes), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
# Phase 1: Train top layers
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_PHASE1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

print('Phase 1: Training top layers...')
history1 = model.fit(train_gen, validation_data=val_gen,
                     epochs=EPOCHS_PHASE1, callbacks=callbacks, verbose=1)

In [ ]:
# Phase 2: Fine-tune entire model
base_model.trainable = True
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_PHASE2),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 2: Fine-tuning entire model...')
history2 = model.fit(train_gen, validation_data=val_gen,
                     epochs=EPOCHS_PHASE2, callbacks=callbacks, verbose=1)

In [ ]:
# Evaluate
val_loss, val_acc = model.evaluate(val_gen, verbose=0)
print(f'Validation accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)')

In [ ]:
# Save model and class names
os.makedirs('model_output', exist_ok=True)

model.save_weights('model_output/model.weights.h5')
print('Weights saved to model_output/model.weights.h5')

with open('model_output/class_names.json', 'w') as f:
    json.dump(classes, f, indent=2)
print('Class names saved to model_output/class_names.json')
print('Class order:', classes)

In [ ]:
# Download the trained files
files.download('model_output/model.weights.h5')
files.download('model_output/class_names.json')